# Wood Model Training on Colab GPU (v3)

**Goal:** train the wood SurfaceCNN to >80% per-class accuracy.

## What changed in v3 (vs v2 which got 67.3% / clean 74 / smudged 61 / cracked 67)

- **Back to hybrid 6-ch input** - v1 hybrid had clean=84%, which we want to keep.
- **`--extra_block`** - 5-block CNN (3M params vs 1.2M). v2 train_acc plateaued at 76% which means the 4-block net was too shallow to fit the data.
- **`--balanced_sampler`** - oversamples smudged/cracked so every batch has roughly equal classes, without distorting the per-sample gradient like a heavy class boost would.
- **`--label_smoothing 0.1`** - softens hard targets so the model isn't penalised for honest uncertainty between similar wood classes.
- **Softer LR scheduler** (built into train.py): patience 15 / factor 0.7 / min_lr 1e-5. v2 had patience 7 / factor 0.5 which killed the LR 8x by epoch 80 and stalled learning.
- **Modest class boosts** (1.2 each on smudged + cracked) instead of v2's 1.5 on smudged alone - keeps clean class healthy.

## Before running

1. **On your laptop**, regenerate the zip: `python prep_wood_for_colab.py` (produces `wood_data.zip`, ~13 MB)
2. **In Colab**: Runtime -> Change runtime type -> **T4 GPU** -> Save
3. Run all cells. Pick `wood_data.zip` when the upload prompt opens.
4. Training takes ~30-40 min on T4 (longer than v2 due to extra_block + 100 epochs).

In [ ]:
# 1. Verify we have a GPU.
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU detected. Runtime -> Change runtime type -> T4 GPU, then Run all again.')

In [ ]:
# 2. Upload wood_data.zip (the file the prep script produced on your laptop).
from google.colab import files
uploaded = files.upload()
import os
assert 'wood_data.zip' in uploaded, f"Expected wood_data.zip, got: {list(uploaded)}"
print('uploaded size:', os.path.getsize('wood_data.zip') / 1024 / 1024, 'MB')

In [ ]:
# 3. Extract zip (code + data) into the working directory.
import zipfile, os
with zipfile.ZipFile('wood_data.zip') as zf:
    zf.extractall('.')
print('contents of cwd:')
for entry in sorted(os.listdir('.')):
    print(' ', entry)
for cls in ['clean', 'smudged', 'cracked']:
    n = len(os.listdir(f'data/wood/{cls}'))
    print(f'data/wood/{cls}: {n} images')
    assert n > 50, f'Too few wood/{cls} images ({n}) - did the prep script run successfully?'

In [ ]:
# 4. Install dependencies (Colab usually has most, but make sure).
!pip install -q opencv-python-headless scikit-learn matplotlib

In [ ]:
# 5. Smoke test: confirm wood preprocessing produces 6-channel tensors (hybrid mode).
import importlib, dataset; importlib.reload(dataset)
ds = dataset.SurfaceDataset('data/wood', 'wood', 'train', augment=True, seed=42)
t, lbl = ds[0]
print('sample tensor:', t.shape, 'dtype:', t.dtype, 'min/max:', float(t.min()), float(t.max()))
print('class counts:', ds.class_counts())
assert t.shape[0] == 6, f'Expected 6 channels (hybrid), got {t.shape[0]}'

In [ ]:
# 6. Train wood (v3 config). Flags explained:
#    --extra_block         : add 5th conv block (256->512), bumps params 1.2M -> 3M.
#    --balanced_sampler    : equal class frequency per batch (helps smudged + cracked).
#    --label_smoothing 0.1 : softens hard targets, reduces overconfident clean preds.
#    --smudge_boost 1.2    : mild extra weight on smudged loss.
#    --cracked_boost 1.2   : mild extra weight on cracked loss.
#    --dropout_fc 0.5      : default FC dropout; with extra capacity we need it.
#    --dropout2d_scale 0.8 : slightly lower conv dropout to let model fit better.
#    train.py wood defaults: epochs=100, lr=0.0005, patience=25 (scheduler patience 15).
!python train.py --surface wood \
    --extra_block \
    --balanced_sampler \
    --label_smoothing 0.1 \
    --smudge_boost 1.2 \
    --cracked_boost 1.2 \
    --dropout_fc 0.5 \
    --dropout2d_scale 0.8

In [ ]:
# 7. Evaluate on the test set with TTA (4-view averaging) for an extra accuracy bump.
!python evaluate.py --surface wood --tta

In [ ]:
# 8. Download wood_best.pth + the training log straight to your laptop.
from google.colab import files
files.download('outputs/wood_best.pth')
files.download('outputs/wood_train.log')
import os
if os.path.isfile('outputs/wood_confusion_matrix.png'):
    files.download('outputs/wood_confusion_matrix.png')